# OpenMonitor AIA ROI light-curve notebook

**EN.** Use this notebook to read an AIA ROI result exported by OpenMonitor Solar Lab. You can paste a result id, a job id, or a Solar Lab share link, or upload the downloaded files manually.

**ES.** Usa este cuaderno para leer un resultado AIA ROI exportado por OpenMonitor Solar Lab. Puedes pegar un result id, un job id o un enlace compartido, o subir los archivos manualmente.

**PT.** Use este notebook para ler um resultado AIA ROI exportado pelo OpenMonitor Solar Lab. Voce pode colar um result id, job id ou link compartilhado, ou enviar os arquivos manualmente.

Recommended files from the web tool: `lightcurves.fits` plus `metadata.json`. CSV/ECSV also work. Movies and PNGs are optional visual context.


## 1. Runtime setup

This installs only common scientific Python packages if Colab does not already provide them.


In [ ]:
import importlib.util
import subprocess
import sys

packages = ["astropy", "pandas", "matplotlib", "numpy", "requests"]
missing = [pkg for pkg in packages if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("Runtime ready.")


## 2. Choose input

This notebook is preloaded with a public demo run from Solar OpenMonitor:

- Recovery code: `aia_963db1ee7c6b49b9a9`
- Event: X1.6 flare, 2014-10-22
- AIA channel: 1700 Å
- Cadence: 24 s

Run all cells to download the cached OpenMonitor result and inspect the FITS/CSV/metadata exports.

You can replace `OPENMONITOR_RESULT` with any of these:

- a Solar Lab share link containing `aiaJob=...`
- an AIA recovery code such as `aia_abc123...`
- a result id/cache key such as `b03c1a0b004f04f0d0b8`
- an API result URL such as `https://api.openmonitor.org/api/solar/aia/results/<id>.json`

Leave `OPENMONITOR_RESULT` empty if you prefer manual upload.


In [ ]:
OPENMONITOR_RESULT = "aia_963db1ee7c6b49b9a9"  # paste a result id, job id, or share link here

# Optional: compare multiple AIA runs, for example one result per wavelength.
# Leave empty to use OPENMONITOR_RESULT above.
OPENMONITOR_RESULTS = [
    # "aia_...",  # 171 Å run
    # "aia_...",  # 304 Å run
]

API_BASE = "https://api.openmonitor.org"
WORKDIR = "openmonitor_aia_result"

# If both OPENMONITOR_RESULT and OPENMONITOR_RESULTS are empty in Colab, the next cells will ask you to upload files.


## 3. Fetch from OpenMonitor or upload files

The notebook downloads only processed result artifacts. It does not fetch the private/raw server-side JSOC cache.


In [ ]:
from pathlib import Path
from urllib.parse import parse_qs, urlparse
import hashlib
import json
import re
import requests


def parse_openmonitor_reference(value):
    value = str(value or "").strip()
    if not value:
        return None, None
    parsed = urlparse(value)
    if parsed.scheme and parsed.netloc:
        query = parse_qs(parsed.query)
        if query.get("aiaJob"):
            return "job", query["aiaJob"][0]
        parts = [part for part in parsed.path.split("/") if part]
        if "results" in parts:
            token = parts[parts.index("results") + 1]
            return "result", token.replace(".json", "")
    parts = [part for part in parsed.path.split("/") if part]
    if "results" in parts:
        token = parts[parts.index("results") + 1]
        return "result", token.replace(".json", "")
    if value.startswith("aia_"):
        return "job", value
    return "result", value.replace(".json", "")


def safe_ref_label(reference, index):
    kind, token = parse_openmonitor_reference(reference)
    token = token or hashlib.sha1(str(reference).encode("utf-8")).hexdigest()[:10]
    token = re.sub(r"[^A-Za-z0-9_.-]+", "_", token).strip("_")
    return f"run_{index:02d}_{token[:32]}"


def result_id_from_job(job_id):
    url = f"{API_BASE}/api/solar/aia/jobs/{job_id}"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    job = response.json()
    if job.get("status") != "complete":
        raise RuntimeError(f"Job is not complete yet: {job.get('status')} - {job.get('errorMessage')}")
    result_url = job.get("resultUrl") or ""
    kind, token = parse_openmonitor_reference(result_url)
    if kind != "result" or not token:
        raise RuntimeError(f"Could not resolve result id from job response: {job}")
    return token


def download_url(url, path, timeout=90):
    path = Path(path)
    response = requests.get(url, timeout=timeout)
    if response.status_code == 404:
        return False
    response.raise_for_status()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(response.content)
    return True


def artifact_url(result_id, relative_path=None):
    if relative_path is None:
        return f"{API_BASE}/api/solar/aia/results/{result_id}.json"
    return f"{API_BASE}/api/solar/aia/results/{result_id}/{relative_path}"


def fetch_openmonitor_result(reference, outdir=WORKDIR):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    kind, token = parse_openmonitor_reference(reference)
    if kind == "job":
        token = result_id_from_job(token)
    if kind is None or not token:
        raise ValueError("Paste an OpenMonitor result id, job id, API URL, or share link.")

    downloaded = []
    if download_url(artifact_url(token), outdir / "result.json"):
        downloaded.append("result.json")

    candidates = [
        "lightcurves.fits", "lightcurves.ecsv", "lightcurves.csv", "metadata.json",
        "lightcurves.png", "roi_overlay.png", "movie.gif", "manifest.json",
        "science_bundle_manifest.json", "frames_manifest.json", "request.json",
        "roi_movies/ROI_A.gif", "roi_movies/ROI_B.gif", "roi_movies/ROI_C.gif", "roi_movies/ROI_D.gif",
    ]
    for rel in candidates:
        if download_url(artifact_url(token, rel), outdir / rel):
            downloaded.append(rel)

    print(f"Resolved result id: {token}")
    print(f"Downloaded {len(downloaded)} files to {outdir.resolve()}")
    for name in downloaded:
        print(" -", name)
    return outdir


def upload_openmonitor_files(outdir=WORKDIR):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    try:
        from google.colab import files
    except Exception as exc:
        raise RuntimeError("Manual upload is available in Google Colab. Otherwise place files in WORKDIR.") from exc
    uploaded = files.upload()
    for name, data in uploaded.items():
        path = outdir / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(data)
    print(f"Uploaded {len(uploaded)} files to {outdir.resolve()}")
    return outdir


def configured_references():
    refs = [str(item).strip() for item in OPENMONITOR_RESULTS if str(item).strip()]
    if refs:
        return refs
    single = str(OPENMONITOR_RESULT or "").strip()
    return [single] if single else []


references = configured_references()
if references:
    result_dirs = []
    if len(references) == 1:
        result_dirs.append(fetch_openmonitor_result(references[0], WORKDIR))
    else:
        root = Path(WORKDIR)
        root.mkdir(parents=True, exist_ok=True)
        for index, reference in enumerate(references, 1):
            print(f"\n=== Fetching run {index}/{len(references)} ===")
            result_dirs.append(fetch_openmonitor_result(reference, root / safe_ref_label(reference, index)))
else:
    result_dirs = [upload_openmonitor_files(WORKDIR)]

result_dir = result_dirs[0]  # Backward-compatible alias for the first run.
print("\nRun directories:")
for item in result_dirs:
    print(" -", item)


## 4. Read the table

Priority is FITS, then ECSV, then CSV. FITS/ECSV preserve richer metadata; CSV is useful for spreadsheets.


In [ ]:
from astropy.table import Table
from IPython.display import display
import numpy as np
import pandas as pd


def first_existing(directory, names):
    directory = Path(directory)
    for name in names:
        path = directory / name
        if path.exists():
            return path
    return None


def load_run(directory, index=1):
    directory = Path(directory)
    table_path = first_existing(directory, ["lightcurves.fits", "lightcurves.ecsv", "lightcurves.csv"])
    if table_path is None:
        raise FileNotFoundError(f"No lightcurves.fits, lightcurves.ecsv, or lightcurves.csv found in {directory}.")

    table_metadata = {}
    if table_path.suffix.lower() in {".fits", ".fit", ".ecsv"}:
        table = Table.read(table_path)
        table_metadata = dict(table.meta)
        frame = table.to_pandas()
    else:
        frame = pd.read_csv(table_path)

    metadata = dict(table_metadata)
    metadata_path = directory / "metadata.json"
    if metadata_path.exists():
        with metadata_path.open("r", encoding="utf-8") as handle:
            metadata.update(json.load(handle))

    if "UTC" not in frame.columns:
        raise ValueError(f"Expected a UTC column in {table_path}.")
    frame["UTC"] = pd.to_datetime(frame["UTC"].astype(str), utc=True)

    system_columns = {"UTC", "EXPTIME", "QUALITY"}
    curves = [
        col for col in frame.columns
        if col not in system_columns and not col.endswith("_sat_frac") and not col.endswith("_saturated")
    ]
    source = metadata.get("source", {}) if isinstance(metadata.get("source"), dict) else {}
    processing = metadata.get("processing", {}) if isinstance(metadata.get("processing"), dict) else {}
    wavelength = source.get("wavelength_angstrom") or source.get("wavelengthAngstrom") or "?"
    signal_mode = processing.get("signal_mode") or processing.get("signalMode") or "exported"
    label = f"{wavelength} Å · {signal_mode}"
    return {
        "index": index,
        "dir": directory,
        "table_path": table_path,
        "df": frame,
        "metadata": metadata,
        "roi_columns": curves,
        "source": source,
        "processing": processing,
        "label": label,
        "wavelength": wavelength,
    }


runs = [load_run(path, index + 1) for index, path in enumerate(result_dirs)]

# Backward-compatible aliases for single-run cells and custom user edits.
primary = runs[0]
df = primary["df"]
metadata = primary["metadata"]
table_path = primary["table_path"]
roi_columns = primary["roi_columns"]

for run in runs:
    print(f"Loaded run {run['index']}: {run['label']}")
    print(f"  table: {run['table_path']}")
    print(f"  rows: {len(run['df'])}")
    print(f"  ROI curve columns: {run['roi_columns']}")

print("\nPreview of first run:")
display(df.head())


## 5. Inspect scientific metadata

OpenMonitor exports ROI coordinates in image-plane helioprojective arcsec, not heliographic latitude/longitude. The table is intended for quick-look and relative analysis unless you apply your own AIA calibration/degradation corrections.


In [ ]:
def get_nested(data, *keys, default=None):
    cur = data
    for key in keys:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur


def print_run_metadata(run):
    frame = run["df"]
    metadata = run["metadata"]
    source = metadata.get("source", {}) if isinstance(metadata.get("source"), dict) else {}
    calibration = metadata.get("calibration", {}) if isinstance(metadata.get("calibration"), dict) else {}
    quality = metadata.get("quality", {}) if isinstance(metadata.get("quality"), dict) else {}
    pixel_scale = metadata.get("pixel_scale", {}) if isinstance(metadata.get("pixel_scale"), dict) else {}
    processing = metadata.get("processing", {}) if isinstance(metadata.get("processing"), dict) else {}
    roi_meta = metadata.get("roi", {}) if isinstance(metadata.get("roi"), dict) else {}

    print(f"\n=== Run {run['index']}: {run['label']} ===")
    print("Source")
    print("  instrument:", source.get("instrument", "SDO/AIA"))
    print("  series:", source.get("series"))
    print("  wavelength_angstrom:", source.get("wavelength_angstrom"))
    print("  cadence_seconds:", source.get("cadence_seconds"))
    print("  UTC span:", frame["UTC"].iloc[0], "->", frame["UTC"].iloc[-1])

    print("Processing")
    print("  signal_mode:", processing.get("signal_mode"))
    print("  normalize_exposure:", processing.get("normalize_exposure"))
    print("  background:", processing.get("background"))

    print("Calibration state")
    for key in ["DATALEVEL", "AIAPREP", "PSFDECON", "DEGRADCORR", "EXPNORM"]:
        print(f"  {key}:", calibration.get(key))
    if calibration.get("calibration_note"):
        print("  note:", calibration.get("calibration_note"))

    print("Frame quality")
    print("  EXPTIME column present:", "EXPTIME" in frame.columns)
    print("  QUALITY column present:", "QUALITY" in frame.columns)
    print("  quality summary:", quality)
    if metadata.get("quality_note"):
        print("  note:", metadata.get("quality_note"))

    print("Pixel scale")
    print(pixel_scale)

    print("ROI boxes")
    for box in roi_meta.get("boxes", []):
        print(f"  {box.get('label') or box.get('id')}: center={box.get('center_arcsec')} size={box.get('size_arcsec')} bounds={box.get('bounds_arcsec')} pixels={box.get('pixel_count')}")

    attribution = metadata.get("attribution", {}) if isinstance(metadata.get("attribution"), dict) else {}
    if attribution:
        print("Attribution")
        print(attribution.get("ready_to_paste"))


for run in runs:
    print_run_metadata(run)

# Backward-compatible aliases for the first run.
source = primary["metadata"].get("source", {}) if isinstance(primary["metadata"].get("source"), dict) else {}
calibration = primary["metadata"].get("calibration", {}) if isinstance(primary["metadata"].get("calibration"), dict) else {}
quality = primary["metadata"].get("quality", {}) if isinstance(primary["metadata"].get("quality"), dict) else {}
pixel_scale = primary["metadata"].get("pixel_scale", {}) if isinstance(primary["metadata"].get("pixel_scale"), dict) else {}
processing = primary["metadata"].get("processing", {}) if isinstance(primary["metadata"].get("processing"), dict) else {}
roi_meta = primary["metadata"].get("roi", {}) if isinstance(primary["metadata"].get("roi"), dict) else {}
attribution = primary["metadata"].get("attribution", {}) if isinstance(primary["metadata"].get("attribution"), dict) else {}


## 6. Plot ROI light curves

Default plot uses the exported values exactly as provided. If `EXPNORM=true`, the values are exposure-normalized intensities, approximately DN/s. If the selected signal was flare excess, the baseline was already subtracted by OpenMonitor.


In [ ]:
import datetime as dt
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

# absolute: plot exported values in their real units, usually DN/s/pixel when EXPNORM=true.
# peak_normalized: divide each curve by its own peak absolute value for shape/timing comparison.
# local_excess: subtract the first LOCAL_BASELINE_MINUTES from each curve before plotting.
PLOT_MODE = "absolute"  # "absolute", "peak_normalized", or "local_excess"
LOCAL_BASELINE_MINUTES = 2
OUTPUT_PLOT = "openmonitor_aia_roi_lightcurve.png"


def calibration_for(run):
    metadata = run.get("metadata", {})
    return metadata.get("calibration", {}) if isinstance(metadata.get("calibration"), dict) else {}


def y_for_plot(run, col):
    frame = run["df"]
    y = pd.to_numeric(frame[col], errors="coerce").astype(float)
    if PLOT_MODE == "local_excess":
        t0 = frame["UTC"].iloc[0]
        mask = frame["UTC"] <= t0 + pd.Timedelta(minutes=LOCAL_BASELINE_MINUTES)
        y = y - y[mask].mean()
    elif PLOT_MODE == "peak_normalized":
        peak = np.nanmax(np.abs(y))
        if peak and np.isfinite(peak):
            y = y / peak
    return y


if not any(run["roi_columns"] for run in runs):
    raise ValueError("No ROI curve columns found.")

fig, ax = plt.subplots(figsize=(10.5, 5.0))
colors = plt.get_cmap("tab10")
line_index = 0

for run in runs:
    frame = run["df"]
    for col in run["roi_columns"]:
        label = f"{run['label']} · {col}" if len(runs) > 1 else col
        ax.plot(
            frame["UTC"],
            y_for_plot(run, col),
            linewidth=1.9,
            label=label,
            color=colors(line_index % 10),
        )
        line_index += 1

if PLOT_MODE == "peak_normalized":
    unit = "normalized to each curve peak"
elif PLOT_MODE == "local_excess":
    unit = f"local excess after first {LOCAL_BASELINE_MINUTES} min"
else:
    first_calibration = calibration_for(runs[0])
    unit = "DN/s/pixel" if first_calibration.get("EXPNORM") else "exported units"

ax.set_xlabel("Time (UTC)")
ax.set_ylabel(f"ROI mean intensity ({unit})")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=dt.timezone.utc))
ax.legend(frameon=False, loc="best", fontsize=8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.autofmt_xdate(rotation=0, ha="center")
fig.tight_layout()
fig.savefig(OUTPUT_PLOT, dpi=180, bbox_inches="tight")
plt.show()
print("Saved plot:", OUTPUT_PLOT)
print("Plot mode:", PLOT_MODE)


## 7. Display visual context if available

`ROI_A.gif`, `ROI_B.gif`, etc. are usually the clearest movies for presentations. `movie.gif` is the wider context cutout.


In [ ]:
from IPython.display import Image, Markdown, display


def show_if_exists(path, title):
    path = Path(path)
    if path.exists():
        display(Markdown(f"**{title}**"))
        display(Image(filename=str(path)))


for run in runs:
    prefix = f"Run {run['index']} · {run['label']}"
    base = Path(run["dir"])
    show_if_exists(base / "lightcurves.png", f"{prefix}: server plot")
    show_if_exists(base / "roi_overlay.png", f"{prefix}: ROI overlay image")
    for gif in sorted((base / "roi_movies").glob("*.gif")):
        show_if_exists(gif, f"{prefix}: ROI movie {gif.name}")
    show_if_exists(base / "movie.gif", f"{prefix}: context movie")


## 8. Export a clean table

This creates a compact CSV for your own analysis. It keeps `UTC`, ROI curves, and `EXPTIME`/`QUALITY` when present.


In [ ]:
EXPORT_CSV = "openmonitor_aia_clean_table.csv"
EXPORT_METADATA = "openmonitor_aia_metadata_summary.json"
DOWNLOAD_OUTPUTS = False  # set True in Colab if you want automatic downloads


def clean_table_for(run):
    frame = run["df"]
    keep = ["UTC"] + [col for col in ["EXPTIME", "QUALITY"] if col in frame.columns] + run["roi_columns"]
    clean = frame[keep].copy()
    clean["UTC"] = clean["UTC"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    if len(runs) > 1:
        prefix = str(run["wavelength"]).replace(".", "_")
        rename = {col: f"{prefix}A_{col}" for col in run["roi_columns"]}
        clean = clean.rename(columns=rename)
    return clean


if len(runs) == 1:
    clean = clean_table_for(runs[0])
else:
    clean = None
    for run in runs:
        part = clean_table_for(run)
        if clean is None:
            clean = part
        else:
            clean = pd.merge(clean, part, on="UTC", how="outer", suffixes=("", f"_run{run['index']}"))
    clean = clean.sort_values("UTC")

clean.to_csv(EXPORT_CSV, index=False)

summary = {
    "runs": [
        {
            "label": run["label"],
            "directory": str(run["dir"]),
            "source": run["source"],
            "processing": run["processing"],
            "roi_columns": run["roi_columns"],
            "metadata": {
                "calibration": run["metadata"].get("calibration", {}),
                "quality": run["metadata"].get("quality", {}),
                "pixel_scale": run["metadata"].get("pixel_scale", {}),
                "roi": run["metadata"].get("roi", {}),
                "attribution": run["metadata"].get("attribution", {}),
            },
        }
        for run in runs
    ],
    "plot_mode": PLOT_MODE,
    "table_columns": list(clean.columns),
}
Path(EXPORT_METADATA).write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Wrote:", EXPORT_CSV)
print("Wrote:", OUTPUT_PLOT)
print("Wrote:", EXPORT_METADATA)

if DOWNLOAD_OUTPUTS:
    try:
        from google.colab import files
        for name in [EXPORT_CSV, OUTPUT_PLOT, EXPORT_METADATA]:
            files.download(name)
    except Exception as exc:
        print("Automatic download is only available in Colab:", exc)


## Notes for citation and reuse

Use the attribution block printed above when describing the product. The OpenMonitor table is derived from SDO/AIA level-1 cutouts obtained via JSOC. For absolute photometry or cross-date comparison, apply your own AIA calibration/degradation correction and inspect/filter `QUALITY` according to your science case.
